# 02 · Experiments

**Purpose:** train and test models on full MIMIC-IV with a GPU (Runtime → Change runtime type → GPU).
Each teammate runs **their own share of the configs** and writes **their own CSV**, so runs happen in
parallel and nobody overwrites anyone else's results.

| Person | Configs |
|---|---|
| Avni | TODO |
| Pawel | TODO |
| Surabhi | TODO |

Config names are listed in `CONFIGS` in `examples/mimic4_mortality_tpc.py`.

> **Data rules:** never print, display or save raw MIMIC-IV rows. Only aggregate counts and scores.
> Don't ask Colab's AI assistant about cells that touch the data. Never commit data or data outputs to GitHub.

## Setup (run every session)

Change `DRIVE_FOLDER` to where the team's shared folder appears in **your** Drive.
A folder someone shared with you shows under "Shared with me": right-click it and choose
**Organize → Add shortcut** to My Drive so Colab can see it.

In [ ]:
# --- Settings: edit these ---
DRIVE_FOLDER = "/content/drive/MyDrive/DL4H_TPC"  # shared team folder
BRANCH = "tpc-dev"
REPO_URL = "https://github.com/avnimohan9/PyHealth.git"

import os, sys
assert sys.version_info >= (3, 12), f"PyHealth needs Python 3.12+, this runtime has {sys.version}"

from google.colab import drive
drive.mount("/content/drive")

# Everything lives in the shared Drive folder
MIMIC4_ROOT = f"{DRIVE_FOLDER}/mimiciv/2.2"   # MIMIC-IV data (hosp/, icu/)
CACHE_DIR = f"{DRIVE_FOLDER}/cache"           # PyHealth's processed copy of the data
RESULTS_DIR = f"{DRIVE_FOLDER}/results"       # results spreadsheets
for d in (CACHE_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Get the latest code from GitHub (clone the first time, pull after that)
if os.path.exists("/content/PyHealth"):
    !git -C /content/PyHealth checkout -q -- .   # drop copies from last run; restored below
    !git -C /content/PyHealth pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} /content/PyHealth
%cd /content/PyHealth
!git log --oneline -1
!pip install -q -e .
# If "import pyhealth" fails right after the first install: Runtime -> Restart session, then rerun setup.

In [ ]:
# Use the latest code built in the model dev notebook (saved in Drive), if there is any
import shutil
for rel in ["pyhealth/models/tpc.py", "tests/core/test_tpc.py", "examples/mimic4_mortality_tpc.py"]:
    src = f"{DRIVE_FOLDER}/code/{rel}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/PyHealth/{rel}")
        print("using the dev notebook's version of", rel)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (switch the runtime to GPU)")

## Your share
Set your name and configs. Results append to your own CSV in the shared folder.

In [ ]:
MY_NAME = "avni"                      # TODO
MY_CONFIGS = ["tpc", "lstm"]          # TODO: your share from the table above
MY_CSV = f"{RESULTS_DIR}/results_{MY_NAME}.csv"
CONFIG_ARGS = " ".join(MY_CONFIGS)

> **Cache:** the first time anyone loads the data, PyHealth builds a processed copy in
> `DL4H_TPC/cache/` (slow, once). After that every session and every teammate reuses it. Let **one
> person** build it first with notebook 01; two people building it at the same time can clash.

## 1. Smoke test (2 epochs, 1 seed)
Run this first to catch errors quickly.

In [ ]:
!python examples/mimic4_mortality_tpc.py --root "{MIMIC4_ROOT}" --cache-dir "{CACHE_DIR}" --smoke --configs {CONFIG_ARGS} --csv /content/smoke_{MY_NAME}.csv

## 2. Full runs
Slow. Safe to rerun after a disconnect: finished runs already in your CSV are skipped.

In [ ]:
!python examples/mimic4_mortality_tpc.py --root "{MIMIC4_ROOT}" --cache-dir "{CACHE_DIR}" --configs {CONFIG_ARGS} --csv {MY_CSV}